In [2]:
import json
import os
from pathlib import Path

import geopandas as gpd
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LightSource
from matplotlib.lines import Line2D
from pyproj import Transformer
from scipy import ndimage

SP = Path(__file__).parent
DATA = Path("../data")
OUT = SP / "poster"

C_RT, C_NT, C_GN = "#eb6834", "#1baf7a", "#7b3fa0"
C_WATER, C_WATER_EDGE, C_WATER_TXT = "#b9d6ee", "#6d9cc4", "#3f6f99"
INK, INK2 = "#1f1f1e", "#55554f"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11, "axes.edgecolor": INK2,
                     "xtick.color": INK2, "ytick.color": INK2, "text.color": INK, "figure.facecolor": "white"})
halo = [pe.withStroke(linewidth=2.6, foreground="white")]
REGION = (175.93, 176.42, -38.748, -38.482)

# ---------------- data ----------------
site = json.load(open(DATA / "site.json"))["drill_site"]
DRILL = (site["lon"], site["lat"])
to_wgs = Transformer.from_crs(2193, 4326, always_xy=True)
merc = []
for line in (DATA / "RTK_NTM_Station_Coordinates.dat").read_text().splitlines()[2:]:
    p = line.split()
    if len(p) >= 4:
        merc.append((" ".join(p[:-3]), p[-3], float(p[-2]), float(p[-1])))
merc = pd.DataFrame(merc, columns=["field", "name", "e", "n"])
merc["lon"], merc["lat"] = to_wgs.transform(merc.e.values, merc.n.values)

gn = pd.read_csv(SP / "geonet/stations.csv")
gs = pd.read_csv(SP / "geonet/sensors.csv")
cur = gs[gs["End Date"].str.startswith("9999")]
seis = cur[cur.Model.str.contains("LE-3D|Trillium|CMG|STS|L4|Guralp", case=False)]
geonet = gn[gn["End Date"].str.startswith("9999") & gn.Station.isin(seis.Station)
            & gn.Longitude.between(*REGION[:2]) & gn.Latitude.between(*REGION[2:])].copy()

os.environ.setdefault("SHAPE_RESTORE_SHX", "YES")


def water_polys(path):
    g = gpd.read_file(path).set_crs(2193, allow_override=True).to_crs(4326)
    return [np.c_[p.exterior.xy] for geom in g.geometry for p in getattr(geom, "geoms", [geom])]


water = water_polys(SP / "lakeshp/nz-lake-polygons-topo-150k.shp") + \
    water_polys(SP / "rivershp/nz-river-polygons-topo-150k.shp")

dem = np.load(SP / "dem/dem_xwide.npz")
elev, dlon, dlat = dem["elev"], dem["lon"], dem["lat"]
KY = 111.32
KX = KY * np.cos(np.radians(DRILL[1]))
shade = LightSource(azdeg=315, altdeg=40).hillshade(elev, vert_exag=2.0, dx=(dlon[1] - dlon[0]) * KX * 1000,
                                                     dy=(dlat[0] - dlat[1]) * KY * 1000)
EXTENT = (dlon[0], dlon[-1], dlat[-1], dlat[0])

# Lake Taupō: the DEM's flat 357 m lake surface, connected to a point in the lake
lab, _ = ndimage.label(np.abs(elev - 357.0) < 0.05)
lake = np.pad(lab == lab[np.argmin(abs(dlat + 38.78)), np.argmin(abs(dlon - 176.0))], 1)
_f, _a = plt.subplots()
_cs = _a.contour(np.r_[dlon[0] - 1e-3, dlon, dlon[-1] + 1e-3], np.r_[dlat[0] + 1e-3, dlat, dlat[-1] - 1e-3],
                 lake.astype(float), [0.5])
water += [seg for seg in _cs.allsegs[0] if len(seg) > 50]
plt.close(_f)
FLAT = float(np.median(shade[lake[1:-1, 1:-1]]))

# ---------------- figure ----------------
W_IN = 11.0
aspect = 1 / np.cos(np.radians(np.mean(REGION[2:])))
h_in = W_IN * 0.86 * (REGION[3] - REGION[2]) * aspect / (REGION[1] - REGION[0]) + 0.9
fig = plt.figure(figsize=(W_IN, h_in))
ax = fig.add_axes([0.07, 0.07, 0.9, 0.9])
ax.imshow(np.clip(1 - shade / FLAT, 0, 1), extent=EXTENT, cmap="Greys", vmin=0, vmax=1.6,
          interpolation="bilinear", zorder=0)
for poly in water:
    ax.fill(poly[:, 0], poly[:, 1], color=C_WATER, ec=C_WATER_EDGE, lw=0.5, zorder=1)
ax.set(xlim=REGION[:2], ylim=REGION[2:])
ax.set_aspect(aspect)
xt = np.arange(175.95, REGION[1], 0.05)
yt = np.arange(-38.70, REGION[3], 0.05)
ax.set_xticks(xt, [f"{v:.2f}°E" for v in xt])
ax.set_yticks(yt, [f"{-v:.2f}°S" for v in yt])
ax.tick_params(labelsize=9, length=3)

# stations: existing = filled
rt, nt = merc[merc.field == "Rotokawa"], merc[merc.field == "Nga Tamariki"]
ax.plot(rt.lon, rt.lat, "^", ms=10, color=C_RT, mec=INK, mew=0.8, ls="none", zorder=5)
ax.plot(nt.lon, nt.lat, "v", ms=10, color=C_NT, mec=INK, mew=0.8, ls="none", zorder=5)
ax.plot(geonet.Longitude, geonet.Latitude, "D", ms=8, color=C_GN, mec=INK, mew=0.8, ls="none", zorder=5)
ax.plot(*DRILL, "*", ms=17, color="#ffd23f", mec=INK, mew=0.9, ls="none", zorder=6)

# station codes, nudged where two stations sit close together
nudge = {"NS11": (-0.0025, 0.0, "right"), "NS14": (0.0025, 0.0, "left"), "RT05": (-0.0025, 0.0, "right"),
         "RT18": (-0.0025, 0.0, "right"), "PRRZ": (-0.0025, 0.0, "right")}
for _, r in pd.concat([merc.rename(columns={"name": "code"})[["code", "lon", "lat"]],
                       geonet.rename(columns={"Station": "code", "Longitude": "lon",
                                              "Latitude": "lat"})[["code", "lon", "lat"]]]).iterrows():
    dx, dy, ha = nudge.get(r.code, (0.0025, 0.0, "left"))
    col = C_GN if r.code in set(geonet.Station) else INK
    ax.text(r.lon + dx, r.lat + dy, r.code, fontsize=7.5, color=col, fontweight="bold", ha=ha, va="center",
            path_effects=halo, zorder=7)

# place names
ax.text(176.245, -38.535, "Ngā Tamariki", fontsize=13, fontweight="bold", path_effects=halo, zorder=7)
ax.text(176.235, -38.592, "Rotokawa", fontsize=13, fontweight="bold", path_effects=halo, zorder=7)
ax.text(175.975, -38.735, "Lake Taupō", fontsize=11, style="italic", color=C_WATER_TXT, path_effects=halo,
        zorder=7)
ax.text(176.187, -38.640, "Lake\nRotokawa", fontsize=8, style="italic", color=C_WATER_TXT, ha="center",
        va="top", path_effects=halo, zorder=7)
ax.text(176.137, -38.616, "Waikato River", fontsize=8.5, style="italic", color=C_WATER_TXT, rotation=32,
        ha="center", path_effects=halo, zorder=7)

# scale bar (5 km) and north arrow
x0, y0 = REGION[1] - 0.085, REGION[2] + 0.012
dl = 5 / KX
ax.plot([x0, x0 + dl / 2], [y0, y0], color=INK, lw=5, solid_capstyle="butt", zorder=9)
ax.plot([x0 + dl / 2, x0 + dl], [y0, y0], color="white", lw=5, solid_capstyle="butt", zorder=9,
        path_effects=[pe.Stroke(linewidth=6.4, foreground=INK), pe.Normal()])
ax.text(x0 + dl / 2, y0 + 0.004, "5 km", ha="center", va="bottom", fontsize=10, path_effects=halo, zorder=9)
ax.annotate("N", xy=(0.965, 0.86), xytext=(0.965, 0.785), xycoords="axes fraction", ha="center", va="center",
            fontsize=12, fontweight="bold", arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.6), zorder=9)

# legend
handles = [Line2D([], [], marker="^", ls="none", ms=10, color=C_RT, mec=INK, label="Mercury, Rotokawa"),
           Line2D([], [], marker="v", ls="none", ms=10, color=C_NT, mec=INK, label="Mercury, Ngā Tamariki"),
           Line2D([], [], marker="D", ls="none", ms=8, color=C_GN, mec=INK, label="GeoNet seismometer"),
           Line2D([], [], marker="*", ls="none", ms=15, color="#ffd23f", mec=INK, label="Drill site")]
ax.legend(handles=handles, loc="lower left", bbox_to_anchor=(0.2, 0.012), fontsize=10, frameon=True,
          framealpha=0.92, edgecolor=INK2, title="Existing seismic stations", title_fontsize=10.5)

# New Zealand inset
nz = gpd.read_file(SP / "coast/nz_land.geojson")
axI = fig.add_axes([0.075, 0.67, 0.14, 0.29])
nz.plot(ax=axI, color="#cfcdc6", edgecolor=INK2, linewidth=0.4)
axI.plot(*DRILL, "s", ms=6, mfc="none", mec="#c0392b", mew=1.6)
axI.set(xlim=(166.0, 179.0), ylim=(-47.6, -34.2), xticks=[], yticks=[], xlabel="", ylabel="")
axI.set_aspect(1 / np.cos(np.radians(-41)))
axI.set_facecolor("white")
for sp in axI.spines.values():
    sp.set_edgecolor(INK)

for ext, kw in (("png", dict(dpi=300)), ("pdf", {})):
    fig.savefig(OUT / f"existing_stations_map.{ext}", facecolor="white", bbox_inches="tight", pad_inches=0.1, **kw)
print("saved", fig.get_size_inches().round(2), "geonet:", ", ".join(geonet.Station))

NameError: name '__file__' is not defined